# 04 - Diagnostics and calibration

Learning dynamics of a run, the direction heads' spread and bias, and an interactive
**calibration explorer**: refit the calibration pipeline on the calibration block with another
interval scale, with or without delta shrinkage, at another miscoverage level, and see test
coverage, interval width, temperature, delta beta, reliability and coverage over time.

In [ ]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

from neural_trade.evaluation.frame import HORIZONS
from neural_trade.metrics.direction_labels import direction_labels_np
from neural_trade.notebook import CalibrationExplorer, load_run_blocks, pick_run
from neural_trade.telemetry.epoch_logger import read_metrics

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
log = pd.DataFrame(read_metrics(run_dir / "metrics.jsonl"))
blocks = load_run_blocks(run_dir, csv_path=CSV_PATH)

## Learning dynamics

In [ ]:
cols = [c for c in ("loss", "val_loss", "val_nll_loss", "val_crps_loss", "val_dir_mcc_h1", "val_gauss_dir_mcc_h1",
                    "val_pred_up_rate_h1", "nonfinite_grad_steps", "grad_global_norm") if c in log]
px.line(log, x="epoch", y=cols, facet_col="variable", facet_col_wrap=3, height=600).update_yaxes(matches=None).show()

## Direction heads on the test block (served = calibrated)

In [ ]:
test, cfg = blocks["test"], blocks["config"]
labels = direction_labels_np(test.y, test.last_close, cfg.DIR_DEADBAND_BPS)
pd.DataFrame({h: {"P(up) std": test.direction_prob[h].std(), "P(up) mean": test.direction_prob[h].mean(),
                  "true up rate (outside deadband)": labels[h][0][labels[h][1]].mean(),
                  "share outside deadband": labels[h][1].mean(),
                  "served delta std ($)": test.delta[h].std(), "realised std ($)": test.y[:, i].std()}
              for i, h in enumerate(HORIZONS)})

## Calibration explorer

Fit on the calibration block, scored on the test block; the run itself is not changed.

In [ ]:
calib = CalibrationExplorer(blocks)
display(calib.widget())
calib.click_refit()   # the run's saved settings first; then change them and press Refit